# 01 - Exploratory Data Analysis
**Kompetisi:** Structural Damage Classification (target `damage_grade` 1-5, metrik Macro F1)

Memahami struktur data, kualitas data, distribusi target, dan hubungan fitur dengan tingkat kerusakan sebelum modeling. Semua gambar disimpan ke `reports/figures/`.

In [ ]:
import sys
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path

sys.path.insert(0, "..")      # agar folder scripts/ bisa di-import dari notebooks/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scripts.preprocess import load_data, split_xy
from scripts.feature_engineering import (
    BASE_NUMERIC_COLS, GEO_COLS, SUPERSTRUCTURE_COL, split_materials,
)
from scripts.visualize import (
    plot_age_vs_damage, plot_categorical_vs_damage, plot_correlation_heatmap,
    plot_damage_by_district, plot_numeric_distributions, plot_superstructure_vs_damage,
    plot_target_distribution,
)

# ---- lokasi data dan output (otomatis: Kaggle atau lokal) ----
ON_KAGGLE = Path("/kaggle/input").exists()
if ON_KAGGLE:
    DATA_DIR = next(Path("/kaggle/input").glob("**/train.csv")).parent
    OUT_DIR = Path("/kaggle/working")
else:                                   # lokal: csv di ../data, output ke root repo
    DATA_DIR = Path("../data")
    OUT_DIR = Path("..")
print("DATA_DIR:", DATA_DIR, "| OUT_DIR:", OUT_DIR)

## 1. Memuat data

In [ ]:
train, test, sample = load_data(DATA_DIR)
print("train:", train.shape, "| test:", test.shape, "| sample:", sample.shape)
X, y = split_xy(train)
train.head()

## 2. Tipe data, nilai kosong, dan jumlah nilai unik

In [ ]:
pd.DataFrame({
    "dtype": train.dtypes.astype(str),
    "n_missing": train.isna().sum(),
    "pct_missing": (train.isna().mean() * 100).round(2),
    "n_unique": train.nunique(),
})

## 3. Distribusi target (`damage_grade`)
Ketidakseimbangan kelas menentukan strategi: Macro F1 memberi bobot sama untuk tiap kelas.

In [ ]:
counts = y.value_counts().sort_index()
display(pd.DataFrame({"jumlah": counts, "persen": (counts / counts.sum() * 100).round(2)}))
plot_target_distribution(y)

## 4. Fitur numerik
Cek outlier dan kode khusus (mis. `age_building` = 999).

In [ ]:
display(train[BASE_NUMERIC_COLS].describe(percentiles=[.01, .5, .99]).T)
print("Korelasi Spearman dengan damage_grade:")
print(train[BASE_NUMERIC_COLS].assign(damage_grade=y.values)
      .corr(method="spearman")["damage_grade"].drop("damage_grade").round(3))
print("\nNilai umur terbanyak (cek kode khusus, mis. 999):")
print(train["age_building"].value_counts().head(8))
plot_numeric_distributions(train)
plot_age_vs_damage(train, y)

## 5. Fitur kategorikal vs tingkat kerusakan

In [ ]:
plot_categorical_vs_damage(train, y)

## 6. Kolom `superstructure`
Cek apakah satu sel berisi beberapa material. Hasilnya menentukan pemisah di `split_materials` (`scripts/feature_engineering.py`).

In [ ]:
print(train[SUPERSTRUCTURE_COL].value_counts(dropna=False).head(15))
n_tok = train[SUPERSTRUCTURE_COL].fillna("unknown").astype(str).map(split_materials).map(len)
print("\nRata-rata material per bangunan:", n_tok.mean().round(2))
plot_superstructure_vs_damage(train, y)

## 7. Fitur geografis

In [ ]:
print(train[GEO_COLS].nunique())
print("Id wilayah di test yang tidak ada di train:",
      {c: int((~test[c].isin(train[c].unique())).sum()) for c in GEO_COLS})
plot_damage_by_district(train, y)

## 8. Korelasi fitur numerik

In [ ]:
plot_correlation_heatmap(X, y)

## 9. Temuan EDA (isi setelah menjalankan notebook)
- Distribusi kelas: ...
- Fitur numerik (outlier / kode khusus): ...
- Kategori yang paling membedakan tingkat kerusakan: ...
- Format `superstructure`: ...
- Kardinalitas fitur geografis dan risiko id baru di test: ...
- Keputusan preprocessing yang diambil berdasarkan temuan di atas: ...